<a href="https://colab.research.google.com/github/AIsa-team/API-use-case-demo/blob/main/Get_a_list_of_videos_from_a_Youtube_channel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import time
import os
from getpass import getpass

# Configuration
API_URL = "https://api.aisa.one/apis/v1/youtube/search"
SEARCH_QUERY = "ycombinator"

def get_api_key():
    """Get API key from environment variable or prompt user."""
    api_key = os.environ.get("AISA_API_KEY")

    if not api_key:
        api_key = getpass("Enter your AISA API key: ")

    if not api_key:
        raise ValueError("AISA API key is required. Set AISA_API_KEY environment variable or enter it when prompted.")

    return api_key

def fetch_ycombinator_channels(api_key):
    """Fetch all channels associated with Y Combinator and return the one with most subscribers."""

    print("Searching for Y Combinator channels...")

    payload = {
        "engine": "youtube",
        "q": SEARCH_QUERY,
        "gl": "US",
        "hl": "en"
    }

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(f"API Error: {response.status_code} - {response.text}")

    data = response.json()
    channels = data.get("channels", [])

    if not channels:
        raise Exception("No channels found for Y Combinator")

    # Print all found channels
    print(f"\nFound {len(channels)} Y Combinator-related channels:")
    print("-" * 60)
    for ch in channels:
        subscribers = ch.get("subscribers", 0)
        # Format subscriber count
        if subscribers >= 1_000_000:
            sub_str = f"{subscribers / 1_000_000:.1f}M"
        elif subscribers >= 1_000:
            sub_str = f"{subscribers / 1_000:.1f}K"
        else:
            sub_str = str(subscribers)

        verified = "✅" if ch.get("is_verified") else ""
        print(f"  • {ch.get('title')} {verified}")
        print(f"    ID: {ch.get('id')}")
        print(f"    Subscribers: {sub_str}")
        print()

    # Find channel with most subscribers
    main_channel = max(channels, key=lambda c: c.get("subscribers", 0))

    print("-" * 60)
    print(f"Selected main channel: {main_channel.get('title')}")
    print(f"Channel ID: {main_channel.get('id')}")
    print(f"Subscribers: {main_channel.get('subscribers'):,}")
    print("-" * 60)

    return main_channel

def fetch_channel_videos(api_key, channel_id, channel_name):
    """Fetch all videos from the specified channel."""

    all_videos = []
    next_page_token = None
    page = 1

    while True:
        print(f"Fetching page {page}...")

        payload = {
            "engine": "youtube",
            "q": channel_name,
            "gl": "US",
            "hl": "en"
        }

        if next_page_token:
            payload["next_page_token"] = next_page_token

        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }

        response = requests.post(API_URL, headers=headers, json=payload)

        if response.status_code != 200:
            print(f"Error: {response.status_code} - {response.text}")
            break

        data = response.json()

        # Extract videos from main results
        videos = data.get("videos", [])

        for video in videos:
            channel = video.get("channel", {})
            vid_channel_id = channel.get("id", "")

            if vid_channel_id == channel_id:
                all_videos.append({
                    "id": video.get("id"),
                    "title": video.get("title"),
                    "link": video.get("link"),
                    "views": video.get("views"),
                    "length": video.get("length"),
                    "published_time": video.get("published_time"),
                    "description": video.get("description", "")[:200],
                    "thumbnail": video.get("thumbnail", {}).get("static", "")
                })

        # Also check sections
        sections = data.get("sections", [])
        for section in sections:
            items = section.get("items", [])
            for video in items:
                channel = video.get("channel", {})
                vid_channel_id = channel.get("id", "")

                if vid_channel_id == channel_id:
                    video_id = video.get("id")
                    if not any(v["id"] == video_id for v in all_videos):
                        all_videos.append({
                            "id": video_id,
                            "title": video.get("title"),
                            "link": video.get("link"),
                            "views": video.get("views"),
                            "length": video.get("length"),
                            "published_time": video.get("published_time"),
                            "description": video.get("description", "")[:200],
                            "thumbnail": video.get("thumbnail", {}).get("static", "")
                        })

        # Check for next page
        pagination = data.get("pagination", {})
        next_page_token = pagination.get("next_page_token")

        if not next_page_token:
            print("No more pages available.")
            break

        page += 1
        time.sleep(1)

        if page > 50:
            print("Reached page limit (50). Stopping.")
            break

    return all_videos

def save_to_json(videos, filename="ycombinator_videos.json"):
    """Save videos to JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(videos, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(videos)} videos to {filename}")

def save_to_csv(videos, filename="ycombinator_videos.csv"):
    """Save videos to CSV file."""
    import csv

    if not videos:
        print("No videos to save.")
        return

    fieldnames = ["id", "title", "link", "views", "length", "published_time", "description", "thumbnail"]

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(videos)

    print(f"Saved {len(videos)} videos to {filename}")

def main():
    print("=" * 60)
    print("Y Combinator Video Fetcher")
    print("(Auto-detects main channel by subscriber count)")
    print("=" * 60)

    # Get API key securely
    api_key = get_api_key()

    # Step 1: Find Y Combinator channels and select the one with most subscribers
    main_channel = fetch_ycombinator_channels(api_key)
    channel_id = main_channel.get("id")
    channel_name = main_channel.get("title")

    # Step 2: Fetch all videos from the main channel
    print(f"\nFetching videos from: {channel_name}")
    print("=" * 60)

    videos = fetch_channel_videos(api_key, channel_id, channel_name)

    print(f"\nTotal videos fetched: {len(videos)}")

    # Remove duplicates by video ID
    unique_videos = list({v["id"]: v for v in videos}.values())

    print(f"Unique videos: {len(unique_videos)}")

    # Save results
    save_to_json(unique_videos)
    save_to_csv(unique_videos)

    # Print summary
    print("\n" + "=" * 60)
    print(f"Summary of first 10 videos from {channel_name}:")
    print("=" * 60)
    for i, video in enumerate(unique_videos[:10], 1):
        print(f"{i}. {video['title']}")
        print(f"   Views: {video['views']} | Length: {video['length']}")
        print(f"   Link: {video['link']}")
        print()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def parse_relative_time(time_str):
    if time_str is None:
        return None

    time_str = time_str.lower().strip()
    now = datetime.now()

    if 'ago' in time_str:
        parts = time_str.split()
        if len(parts) >= 3:
            value = int(parts[0])
            unit = parts[1]

            if 'second' in unit:
                return now - timedelta(seconds=value)
            elif 'minute' in unit:
                return now - timedelta(minutes=value)
            elif 'hour' in unit:
                return now - timedelta(hours=value)
            elif 'day' in unit:
                return now - timedelta(days=value)
            elif 'week' in unit:
                return now - timedelta(weeks=value)
            elif 'month' in unit:
                # Approximate month conversion
                return now - timedelta(days=value * 30)
            elif 'year' in unit:
                # Approximate year conversion
                return now - timedelta(days=value * 365)
    return None

# Apply the function to create a new 'published_datetime' column
yc_videos_df['published_datetime'] = yc_videos_df['published_time'].apply(parse_relative_time)

# Sort the DataFrame by 'published_datetime' in descending order
yc_videos_df_sorted = yc_videos_df.sort_values(by='published_datetime', ascending=False)

# Display the first few rows of the sorted DataFrame
display(yc_videos_df_sorted.head())

In [ ]:
import pandas as pd

yc_videos_df = pd.read_csv('ycombinator_videos.csv')
display(yc_videos_df.head())